# UCS420 Lab Assignment 9 — NLP Text Preprocessing
Enter your roll number and name, then run all cells.

In [ ]:
ROLL_NO = 1024170336
NAME = "Pranati Kondal"

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"

### Dataset generator — do not modify

In [ ]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline — do not modify

In [ ]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [ ]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [ ]:
rows = []
all_tokens = set()

for _, r in df.iterrows():
    res = process_ticket(r["Ticket_Text"])
    all_tokens.update(res["tokens"])
    rows.append({
        "Ticket_ID": r["Ticket_ID"],
        "Sentences": len(res["sentences"]),
        "Tokens": len(res["tokens"]),
        "Punctuation": len(res["punct"]),
        "StopWords": len(res["stops"]),
        "Unique_tokens": len(set(res["tokens"])),
        "Final_Tokens": len(res["final"])
    })

summary = pd.DataFrame(rows)
total = {c: summary[c].sum() for c in summary.columns[1:]}
total["Unique_tokens"] = len(all_tokens)
summary = pd.concat([summary, pd.DataFrame([{"Ticket_ID": "TOTAL", **total}])], ignore_index=True)
display(summary)

selected_id = FOCUS[0]
selected_text = df.loc[df["Ticket_ID"] == selected_id, "Ticket_Text"].iloc[0]
selected_result = process_ticket(selected_text)
print("Selected ticket:", selected_id)
print("Text:", selected_text)
print("Tokens:", selected_result["tokens"])
print("Punctuation tokens:", selected_result["punct"])
print("Stopwords:", selected_result["stops"])
print("Final tokens:", selected_result["final"])

contraction_parts = sorted({token for token in selected_result["final"] if "'" in token or token in {"ca", "n't"}})
print("Contraction-related final tokens in selected ticket:", contraction_parts)

**Answer:** Some final tokens may be fragments such as `n't`, `'s`, or `ca`. NLTK's `word_tokenize()` splits contractions into separate pieces, and stopword removal removes only tokens found in the stopword list. A fragment such as `n't` may remain because it is not removed as a complete stopword. The exact fragments in this dataset are printed with the selected ticket above.

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [ ]:
TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

import re

raw_tickets = df["Ticket_Text"].tolist()
texts_to_check = raw_tickets + TEST_LINES

long_word_pattern = r"\b[A-Za-z]{6,}\b"
number_pattern = r"(?<![A-Za-z0-9])(?:0[xX][0-9A-Fa-f]+|\d{1,3}(?:,\d{3})+(?:\.\d+)*(?:%)?|\d+(?:\.\d+)*(?:%)?)(?![A-Za-z0-9])"
capitalized_pattern = r"\b[A-Z][A-Za-z]*(?:-[A-Za-z]+)*\b"
vowel_pattern = r"\b[AEIOUaeiou][A-Za-z]*\b"

for i, text in enumerate(texts_to_check, start=1):
    print(f"Text {i}")
    print("Words longer than 5 letters:", re.findall(long_word_pattern, text))
    print("Numbers:", re.findall(number_pattern, text))
    print("Capitalized words:", re.findall(capitalized_pattern, text))
    print("Words beginning with a vowel:", re.findall(vowel_pattern, text))
    print()

email_pattern = r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
url_pattern = r"\b(?:https?://|www\.)[^\s<>]*[^\s<>.,!?;:]"
phone_pattern = r"(?<!\w)(?:\+91[\s-]?\d{5}[\s-]?\d{5}|\d{4}-\d{3}-\d{4})(?!\w)"

def replace_identifiers(text):
    text = re.sub(email_pattern, "<EMAIL>", text)
    text = re.sub(url_pattern, "<URL>", text)
    text = re.sub(phone_pattern, "<PHONE>", text)
    return text

print("Identifier replacement on raw tickets")
for ticket in raw_tickets:
    print(replace_identifiers(ticket))

print("\nIdentifier replacement on TEST_LINES")
for line in TEST_LINES:
    print(replace_identifiers(line))

capitalized_count = 0
sentence_initial_words = []
for ticket in raw_tickets:
    for sentence in sent_tokenize(ticket):
        match = re.search(r"\b[A-Za-z][A-Za-z-]*\b", sentence)
        if match and match.group(0)[0].isupper():
            sentence_initial_words.append(match.group(0))
            capitalized_count += 1

print("\nSentence-initial capitalized words:", capitalized_count)
print("Sentence-initial examples:", sentence_initial_words)

**Answer:** Many capitalized words are capitalized simply because they begin a sentence. The code above counts the sentence-initial words in the generated tickets and prints those examples. Other capitalized matches can be acronyms or names, such as `LMS`, `VPN`, or `MATLAB`, which may be capitalized for reasons other than sentence position.

---
## Q3. Custom tokenizer
Write my_tokenize(text) with one RegexpTokenizer pattern that keeps contractions (isn't), hyphenated words (Wi-Fi, state-of-the-art), decimals and versions (95.5, 2.0.1), emails, URLs and phone numbers as single tokens. Run it on TEST_LINES and on your tickets, and compare with word_tokenize.

In [ ]:
from nltk.tokenize import RegexpTokenizer

token_pattern = r"""(?:https?://|www\.)[^\s<>"']*[^\s<>"'.,!?;:]|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}|\+91[\s-]?\d{5}[\s-]?\d{5}|\d{4}-\d{3}-\d{4}|\d{1,3}(?:,\d{3})+(?:\.\d+)*(?:%)?|\d+(?:\.\d+)+(?:%)?|\d+%?|[A-Za-z]+(?:['’][A-Za-z]+)+|[A-Za-z]+(?:-[A-Za-z]+)+|[A-Za-z]+|[^\w\s]"""
tokenizer = RegexpTokenizer(token_pattern)

def my_tokenize(text):
    return tokenizer.tokenize(text)

print("TEST_LINES")
for line in TEST_LINES:
    print("\nText:", line)
    print("Custom tokens:", my_tokenize(line))
    print("NLTK tokens:", word_tokenize(line))

comparison_rows = []
different_tickets = []
for _, row in df.iterrows():
    custom = my_tokenize(row["Ticket_Text"])
    nltk_tokens = word_tokenize(row["Ticket_Text"])
    same = custom == nltk_tokens
    comparison_rows.append({
        "Ticket_ID": row["Ticket_ID"],
        "Custom_token_count": len(custom),
        "NLTK_token_count": len(nltk_tokens),
        "Same_tokenization": same
    })
    if not same:
        different_tickets.append(row["Ticket_ID"])

display(pd.DataFrame(comparison_rows))
print("Tickets tokenized differently:", len(different_tickets), "out of", len(df))
print("Ticket IDs:", different_tickets)

**Answer:** The exact number is printed as “Tickets tokenized differently” in the output above. The custom tokenizer keeps contractions, hyphenated words, decimal/version strings, email addresses, URLs, and phone numbers together. NLTK's default tokenizer may split several of these into smaller tokens, so the token lists differ for tickets containing those patterns.

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [ ]:
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer

unique_final_tokens = sorted({
    token
    for text in df["Ticket_Text"]
    for token in process_ticket(text)["final"]
    if re.fullmatch(r"[a-zA-Z]+", token)
})

porter = PorterStemmer()
lancaster = LancasterStemmer()
regexp_stemmer = RegexpStemmer("ing$", min=5)
lemmatizer = WordNetLemmatizer()

def wordnet_pos(tag):
    if tag.startswith("J"):
        return "a"
    if tag.startswith("V"):
        return "v"
    if tag.startswith("N"):
        return "n"
    if tag.startswith("R"):
        return "r"
    return "n"

tagged_words = nltk.pos_tag(unique_final_tokens)
pos_by_word = {word: tag for word, tag in tagged_words}

stem_rows = []
for word in unique_final_tokens:
    tag = pos_by_word[word]
    stem_rows.append({
        "Word": word,
        "Porter": porter.stem(word),
        "Lancaster": lancaster.stem(word),
        "Regexp_ing": regexp_stemmer.stem(word),
        "POS": tag,
        "WordNet_lemma": lemmatizer.lemmatize(word, wordnet_pos(tag))
    })

stem_table = pd.DataFrame(stem_rows)
different_rows = stem_table[
    (stem_table["Porter"] != stem_table["Lancaster"]) |
    (stem_table["Porter"] != stem_table["Regexp_ing"]) |
    (stem_table["Porter"] != stem_table["WordNet_lemma"])
].reset_index(drop=True)

print("Words ending in -ing in this dataset:", [w for w in unique_final_tokens if w.endswith("ing")])
print("Regexp stemmer uses the suffix 'ing' because it occurs in words such as working/uploading when present.")
display(different_rows)

short_lancaster = different_rows[
    (different_rows["Word"].str.len() >= 5) &
    (different_rows["Lancaster"].str.len() <= 3) &
    (different_rows["Lancaster"] != different_rows["Word"])
]
print("Possible over-stemming examples:")
display(short_lancaster[["Word", "Lancaster"]].head(5))

final_word_set = set(unique_final_tokens)
if "unable" in final_word_set:
    print("Over-stemming example: unable ->", lancaster.stem("unable"))
if "works" in final_word_set and "working" in final_word_set:
    print("Under-stemming example with RegexpStemmer: works ->", regexp_stemmer.stem("works"), "| working ->", regexp_stemmer.stem("working"))
print("The RegexpStemmer only removes the selected -ing suffix and does not handle other endings.")

**Answer:** In this dataset, Lancaster can over-stem `unable` to `un`, which is not a useful word form. The `RegexpStemmer` can under-stem related forms: it leaves `works` unchanged but reduces `working` to `work`, because the rule removes only the `ing` suffix. The actual outputs are printed below the comparison table.

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [ ]:
from collections import Counter
from nltk.util import bigrams
from nltk.stem import PorterStemmer
import numpy as np
import matplotlib.pyplot as plt

s1 = []
s2 = []
s3 = []

for ticket in df["Ticket_Text"]:
    result = process_ticket(ticket)
    s1.extend(result["tokens"])
    s2.extend([token for token in result["tokens"] if not is_punct(token)])
    s3.extend(result["final"])

porter = PorterStemmer()
s4 = [porter.stem(token) for token in s3]

def corpus_stats(name, tokens):
    total = len(tokens)
    unique = len(set(tokens))
    ttr = unique / total if total else 0
    return {"Corpus": name, "Total_tokens": total, "Unique_tokens": unique, "TTR": round(ttr, 4)}

stats = pd.DataFrame([
    corpus_stats("S1: all tokens", s1),
    corpus_stats("S2: no punctuation", s2),
    corpus_stats("S3: final tokens", s3),
    corpus_stats("S4: Porter stems", s4)
])
display(stats)

print("Top 10 words in S1")
display(pd.DataFrame(Counter(s1).most_common(10), columns=["Word", "Frequency"]))

print("Top 10 words in S3")
display(pd.DataFrame(Counter(s3).most_common(10), columns=["Word", "Frequency"]))

freq_s3 = Counter(s3)
ranked = sorted(freq_s3.values(), reverse=True)
ranks = np.arange(1, len(ranked) + 1)
plt.figure(figsize=(7, 4))
plt.plot(np.log(ranks), np.log(ranked), marker="o")
plt.xlabel("log(rank)")
plt.ylabel("log(frequency)")
plt.title("Log(rank) vs log(frequency) for S3")
plt.grid(True)
plt.show()

print("Top 5 bigrams in S1")
display(pd.DataFrame(Counter(bigrams(s1)).most_common(5), columns=["Bigram", "Frequency"]))

print("Top 5 bigrams in S3")
display(pd.DataFrame(Counter(bigrams(s3)).most_common(5), columns=["Bigram", "Frequency"]))

print("TTR changes from S1 to S4 because preprocessing removes punctuation and stopwords, then stemming merges different word forms into the same stem. The total number of tokens and the number of unique tokens both change, so their ratio changes.")

**Answer:** TTR is the number of unique tokens divided by the total number of tokens. From S1 to S4, punctuation and stopwords are removed, and Porter stemming can merge related word forms into one stem. This changes both the numerator and denominator, so TTR changes. The exact TTR values for all four corpora are shown in the table above.